# 09 - זיהוי קהילות (Community Detection)

עד לנקודה זו התייחסנו לרשת התחבורה הציבורית בישראל כאובייקט אחד ושאלנו עד כמה היא שברירית. מחברת זו שואלת שאלה אחרת: **האם הרשת מתפצלת, מעצמה, לגושים המקושרים היטב מבפנים?** אנו מריצים את אלגוריתם **Louvain** למקסום modularity על גרף שכנויות הנסיעות (trip-adjacency) הלא-מכוון והממושקל, מודדים עד כמה החלוקה המתקבלת טובה (**modularity**), מתארים את הקהילות שהוא מוצא, ולאחר מכן מזהים את **תחנות הגשר הבין-קהילתיות** - התחנות ששכניהן מצויים ביותר מקהילה אחת, ולפיכך נושאות את התנועה הנאלצת לחצות גבול בין קהילות.

**שאלת המחקר הנדונה כאן:** האם הרשת היא אוסף של אשכולות אזוריים/מטרופוליניים המקושרים ברפיפות, ולא רשת הומוגנית אחת, ואילו תחנות מחזיקות אשכולות אלה יחד?

### סתירה שמחברת זו נכתבה כדי ליישב

שני תיעודים קודמים של הפרויקט אינם מסכימים זה עם זה: הדוח המסכם בעברית מציין כי Louvain מצא **91** קהילות, בעוד שצינור העיבוד של המצגת מדווח על **73**. אף אחד משני המספרים אינו שגוי במובן של טעות - Louvain הוא *היוריסטיקה חמדנית סטוכסטית*, ולכן מספר הקהילות שהוא מחזיר אינו תכונה קבועה של הגרף. הוא תלוי ב-seed האקראי, בפרמטר הרזולוציה, במימוש, ובגרף המדויק שהוזן לו. לפיכך מחברת זו:

1. חושפת את `LOUVAIN_SEED` ואת `LOUVAIN_RESOLUTION` כקבועים מפורשים בראשה,
2. מדווחת על מה שהריצה **הזו** מפיקה, יחד עם ה-modularity שלה, במקום לצטט מספר הזכור מן העבר,
3. מריצה סריקת seed וסריקת רזולוציה כדי להראות אמפירית עד כמה המספר זז בעוד שהחלוקה עצמה נותרת זהה במהותה,
4. מסבירה, בסעיף 8, אילו מן המנגנונים הללו עשויים לייצר פער בסדר הגודל שנצפה בין שני התיעודים הקודמים.

שום מספר קהילות ושום מזהה קהילה אינם כתובים בקוד או בטקסט בשום מקום להלן; כל מספר וכל תווית מחושבים מן הנתונים בזמן ריצה.

## קלט

* `outputs/nb/02_graph_construction/nodes.csv` - שורה אחת לכל תחנה פעילה, עם `stop_id, stop_name, lat, lon, region, metro`.
* `outputs/nb/02_graph_construction/edges.csv` - שורה אחת לכל מקטע **מכוון** `from_stop, to_stop, trip_frequency`.

שני הקבצים מופקים על ידי מחברת `02_graph_construction`. מחברת זו **אינה** קוראת את קובץ ה-GTFS הגולמי, **אינה** נזקקת לקובץ `stop_times.txt` בנפח 816 MB, ו**אינה** מייבאת דבר מתוך `src/` או מתוך `public_transport_network_research/` - כל הלוגיקה מוטמעת בה עצמה.

## פלט

הכול נכתב תחת `outputs/nb/09_community_detection/`:

* `community_detection_summary.json` - סטטיסטיקות המפתח (seed, רזולוציה, backend, מספר קהילות, modularity, סטטיסטיקות גבול) במילון אחד.
* `tables/community_assignments.csv` - שורה אחת לכל תחנה עם הקהילה שלה לפי Louvain (ולצידה הקהילה שלה לפי label propagation, לשם השוואה).
* `tables/community_summary.csv` - שורה אחת לכל קהילה: גודל, קשתות ומשקל פנימיים/חיצוניים, מרכז מסה, האזור והמטרופולין הדומיננטיים, תחנת ה-hub, והתווית הנגזרת.
* `tables/inter_community_bridges.csv` - כל תחנה שסביבתה משתרעת על יותר מקהילה אחת, מדורגת לפי מספר הקהילות שהיא נוגעת בהן.
* `tables/louvain_stability.csv` - תוצאות סריקת ה-seed וסריקת הרזולוציה (מספר קהילות, modularity, ומידת ההסכמה עם הריצה הראשית).
* `tables/community_detection_summary.csv` - סטטיסטיקות המפתח כטבלה בת שורה אחת.
* `figures/community_map_louvain.png` - כל התחנות משורטטות בקואורדינטות האמיתיות שלהן, צבועות לפי קהילה.
* `figures/community_size_distribution.png` - הקהילות הגדולות ביותר כתרשים עמודות, בתוספת התפלגות דרגה-גודל המלאה.
* `figures/inter_community_bridges_map.png` - התחנות החוצות גבולות קהילה, על גבי המפה.
* `figures/louvain_stability.png` - מספר הקהילות וה-modularity לאורך seeds ורזולוציות.

דבר מחוץ ל-`outputs/nb/09_community_detection/` אינו נוגע; בפרט, התיקיות הקיימות `outputs/tables`, `outputs/figures` ו-`outputs/rail`, המכילות את התוצאות המצוטטות בדוח הכתוב, אינן נכתבות לעולם.

## 1. אתחול סביבת ההרצה

התא שלהלן מאפשר להריץ את המחברת הן על עותק מקומי של המאגר והן על Google Colab. הוא מגדיר את `_ensure(...)`, המתקין באמצעות pip רק את החבילות החסרות בפועל (כך שהרצה חוזרת של המחברת זולה), ואת `find_repo_root()`, המטפסת מן התיקייה הנוכחית כלפי מעלה בחיפוש אחר תיקיית ה-GTFS, ואם אינה מוצאת אותה - משכפלת את המאגר אל `/content`. לאחר מכן הוא מגדיר את `REPO`, `DATA` ו-`OUT` ויוצר את שורש הפלט של המחברת. כל תא מאוחר יותר מסתמך על שלושת הנתיבים הללו, ולכן תא זה חייב לרוץ ראשון. זהו אותו אתחול המשמש בכל שאר המחברות בסדרה, ונשמר זהה במכוון כדי שניתן יהיה להריץ כל אחת מהן באופן עצמאי.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)


## 2. ספריות, תיקיות השלב וקבועים הניתנים לכיוונון

אנו מתקינים ומייבאים את מחסנית הספריות המדעיות, ולאחר מכן מקבעים את מבנה התיקיות של שלב זה. בהתאם למוסכמת הפרויקט, כל מחברת מחזיקה בדיוק בתיקיית פלט אחת: מחברת זו כותבת אל `outputs/nb/09_community_detection/` (עם תת-התיקיות `tables/` ו-`figures/`) וקוראת את השלב הקודם מתוך `outputs/nb/02_graph_construction/`.

בסביבת Python קיימים שני מימושים של Louvain, והם **אינם** מחזירים תמיד את אותה חלוקה: החבילה העצמאית `python-louvain` (המיובאת בשם `community`, זו שבה השתמש הסקריפט המקורי של הפרויקט) והמימוש הנכלל ב-`networkx` החל מגרסה 2.8. אנו מעדיפים את הראשון לשם רציפות עם צינור העיבוד הקודם, ונסוגים לשני אם לא ניתן להתקין את החבילה - הבחירה נרשמת ב-`LOUVAIN_BACKEND` ונכתבת אל קובץ ה-JSON המסכם, משום ש"איזה מימוש" הוא כשלעצמו אחד ההסברים לפער שבין 91 ל-73.

הקבועים מרוכזים כאן כדי שהבודק יוכל לשנות את הניסוי במקום אחד:

* `LOUVAIN_SEED` - ה-seed של מחולל המספרים האקראיים בריצה הראשית. Louvain סורק את הצמתים בסדר אקראי ומפריד שוויונות באקראי, ולכן קבוע זה משנה באמת את התשובה. קיבועו הופך את המחברת לניתנת לשחזור.
* `LOUVAIN_RESOLUTION` - הפרמטר `gamma` בפונקציית המטרה של ה-modularity. הערך `1.0` הוא ההגדרה הקלאסית של Newman-Girvan; ערכים גדולים מ-1 מענישים קהילות גדולות ולכן מחזירים קהילות רבות וקטנות יותר; ערכים קטנים מ-1 מחזירים פחות קהילות וגדולות יותר.
* `SEED_SWEEP` / `RESOLUTION_SWEEP` - שני ניסויי הרגישות של סעיף 8.
* `RUN_LABEL_PROPAGATION` - הרצה נוספת של אלגוריתם קהילות שני, שונה לחלוטין, כבדיקה צולבת.
* יתר הפרמטרים שולטים ברזולוציית האיורים ובמספר השורות המוצג בטבלאות ובתרשימי ה-top-N.

הרצת Louvain על גרף בן כ-30k צמתים / 52k קשתות אורכת כמה שניות לריצה, ולכן הסריקות שלהלן עולות כתריסר ריצות ומסתיימות בכל זאת הרבה פחות מדקה.

In [ ]:
# --- Libraries and stage folders ------------------------------------------
_ensure('pandas', 'numpy', 'networkx', 'matplotlib', 'seaborn', 'python-louvain', 'scikit-learn')

import json
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.05)

# Prefer the standalone python-louvain package (what the original script used);
# fall back to the implementation shipped with networkx.
try:
    import community as community_louvain
    LOUVAIN_BACKEND = 'python-louvain'
except ImportError:
    community_louvain = None
    LOUVAIN_BACKEND = 'networkx'

PREV = OUT / '02_graph_construction'      # read-only: artifacts of notebook 02
STAGE = OUT / '09_community_detection'    # everything this notebook produces
TABLES = STAGE / 'tables'
FIGURES = STAGE / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

# --- Tunable constants ----------------------------------------------------
LOUVAIN_SEED = 42                              # RNG seed of the headline run
LOUVAIN_RESOLUTION = 1.0                       # gamma in the modularity objective
SEED_SWEEP = [0, 1, 7, 42, 123, 2024]          # seeds tried in the stability experiment
RESOLUTION_SWEEP = [0.5, 0.8, 1.0, 1.2, 2.0]   # resolutions tried at the fixed seed
RUN_LABEL_PROPAGATION = True                   # second algorithm, used only as a cross-check
WEIGHT_ATTR = 'weight'                         # edge attribute Louvain optimises over
FIG_DPI = 150                                  # figure resolution; drop to 90 for smaller files
TOP_N = 20                                     # rows shown in every top-N table / bar chart
MAP_ALPHA = 0.6                                # point transparency on the ~30k-point maps
MAP_POINT_SIZE = 3                             # marker size on the community map
ANNOTATE_TOP = 8                               # communities labelled in place on the map

print('previous stage :', PREV)
print('this stage     :', STAGE)
print('Louvain backend:', LOUVAIN_BACKEND)
print(f'seed = {LOUVAIN_SEED}, resolution = {LOUVAIN_RESOLUTION}')


## 3. עיבוד תוויות בעברית

שמות התחנות בקובץ ה-GTFS הישראלי הם בעברית, וכמה מן האיורים שלהלן מציגים אותם (תרשים העמודות של הקהילות מתייג כל קהילה בתחנה העמוסה ביותר שלה). Matplotlib אינה מממשת את האלגוריתם הדו-כיווני (bidirectional) של Unicode, ולכן טקסט מימין לשמאל יוצא הפוך ובלתי קריא. התא שלהלן מבצע monkey-patch חד-פעמי ל-`matplotlib.text.Text.set_text`, כך שכל מחרוזת המכילה תווים בעברית מומרת לסדר תצוגה באמצעות `python-bidi` לפני שהיא משורטטת, ובוחר גופן שיש בו אכן גליפים עבריים (Arial ב-Windows, DejaVu Sans בכל סביבה אחרת). התא הוא אידמפוטנטי - הרצה חוזרת שלו לא תערים patches זה על גבי זה. כל שאר הטקסט במחברת הוא באנגלית, בהתאם לדרישות ההגשה.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()


## 4. טעינת הגרף שהופק במחברת 02

שלב זה תלוי במחברת `02_graph_construction`. במקום לשחזר אובייקט `networkx` מקובץ pickle (קובצי pickle רגישים לגרסאות ואינם קריאים לבודק), אנו טוענים מחדש את שתי טבלאות ה-CSV הפשוטות ששלב 02 מייצא ובונים מהן מחדש את הגרף - קובצי ה-CSV מכילים בדיוק את אותו מידע. זהו שינוי מכוון ביחס לסקריפט המקורי של הפרויקט, שטען את `graph_undirected.pkl`. הפונקציה `find_artifact` סורקת את כל תיקיית השלב הקודם, כך שהיא פועלת בין אם שלב 02 הניח את הטבלאות בשורש התיקייה ובין אם בתוך `tables/`, ומעלה שגיאה מפורשת וברת-פעולה אם קובצי הפלט חסרים.

In [ ]:
# --- Locate the artifacts written by notebook 02 --------------------------
def find_artifact(stage_dir, filename):
    """Return the path of `filename` under a stage folder, or None if absent."""
    if not stage_dir.is_dir():
        return None
    direct = stage_dir / filename
    if direct.exists():
        return direct
    matches = sorted(stage_dir.rglob(filename))
    return matches[0] if matches else None

nodes_path = find_artifact(PREV, 'nodes.csv')
edges_path = find_artifact(PREV, 'edges.csv')
missing = [name for name, p in [('nodes.csv', nodes_path), ('edges.csv', edges_path)] if p is None]
if missing:
    raise FileNotFoundError(
        f"{', '.join(missing)} not found under {PREV} - "
        'run notebook 02_graph_construction first; it writes nodes.csv and edges.csv.'
    )

nodes_df = pd.read_csv(nodes_path, dtype={'stop_id': str}, encoding='utf-8-sig')
edges_df = pd.read_csv(edges_path, dtype={'from_stop': str, 'to_stop': str}, encoding='utf-8-sig')
print(f'nodes: {len(nodes_df):,} rows  <-  {nodes_path}')
print(f'edges: {len(edges_df):,} rows  <-  {edges_path}')
nodes_df.head()


## 5. בנייה מחדש של הגרף הלא-מכוון הממושקל

המודל של הפרויקט הוא **גרף שכנויות נסיעות (trip-adjacency graph)**: צומת הוא תחנה המופיעה בנסיעה אחת לפחות, וקשת מכוונת `u -> v` קיימת כאשר נסיעה כלשהי פוקדת את `v` מיד לאחר `u`; משקל הקשת הוא מספר הנסיעות המשתמשות במקטע זה. הקובץ `edges.csv` שומר את הגרף המכוון הזה. זיהוי קהילות מוגדר עבור גרפים לא-מכוונים, ולכן אנו לוקחים את ההיטל הלא-מכוון: קשת אחת לכל זוג לא-סדור, כאשר משקלי שני כיווני הנסיעה **מחוברים**. משקל מסוכם זה הוא שעליו Louvain ממקסם, ומכאן שמקטע המשורת במאה אוטובוסים בשעה מושך את שני קצותיו לאותה קהילה בעוצמה רבה בהרבה ממקטע המשורת פעמיים ביום - החלוקה עוסקת בעוצמת השירות, ולא רק בטופולוגיה.

תכונות התחנות (שם, קואורדינטות, אזור, מטרופולין) מצורפות מתוך `nodes.csv`. הפונקציות העזר `_num` ו-`_txt` ממירות ערכים ריקים ו-`NaN` באופן בטוח, כך שקואורדינטה חסרה הופכת ל-`None` ולא ל-`NaN` שקט שהיה משורטט בהמשך במיקום חסר משמעות.

In [ ]:
# --- Rebuild the weighted undirected graph ---------------------------------
def _num(value):
    """Coerce to float; return None for blanks, NaN or non-numeric input."""
    if value is None:
        return None
    try:
        f = float(value)
    except (TypeError, ValueError):
        return None
    return None if not np.isfinite(f) else f


def _txt(value):
    """Coerce to a plain string; NaN and None become an empty string."""
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        return ''
    return str(value)


def build_undirected_graph(nodes_df, edges_df):
    """Undirected projection of the directed trip graph; opposite-direction weights summed."""
    attr = {}
    for rec in nodes_df.to_dict('records'):
        attr[str(rec.get('stop_id'))] = {
            'stop_name': _txt(rec.get('stop_name')),
            'lat': _num(rec.get('lat')),
            'lon': _num(rec.get('lon')),
            'region': _txt(rec.get('region')),
            'metro': _txt(rec.get('metro')),
        }

    weight_col = next((c for c in ('trip_frequency', 'weight', 'count')
                       if c in edges_df.columns), None)

    G = nx.Graph()
    for rec in edges_df.to_dict('records'):
        u, v = str(rec['from_stop']), str(rec['to_stop'])
        if u == v:
            continue
        w = int(rec[weight_col]) if weight_col else 1
        if G.has_edge(u, v):
            G[u][v][WEIGHT_ATTR] += w
        else:
            G.add_edge(u, v, **{WEIGHT_ATTR: w})

    default = {'stop_name': '', 'lat': None, 'lon': None, 'region': '', 'metro': ''}
    for n in G.nodes():
        G.nodes[n].update(attr.get(n, default))
    return G


G = build_undirected_graph(nodes_df, edges_df)
total_weight = sum(d[WEIGHT_ATTR] for _, _, d in G.edges(data=True))
unnamed = sum(1 for n in G.nodes() if not G.nodes[n]['stop_name'])
print(f'undirected G : {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')
print(f'total edge weight (trips over all segments): {total_weight:,}')
print(f'nodes with no name attribute: {unnamed:,}')


## 6. צמצום הניתוח לרכיב הקשיר הגדול ביותר

הסקריפט המקורי של הפרויקט הריץ את Louvain על `G.subgraph(max(nx.connected_components(G), key=len))`, כלומר על **הרכיב הקשיר הגדול ביותר בלבד**, ואנו שומרים על בחירה זו. הנימוק הוא שרכיבים מבודדים מהווים באופן טריוויאלי קהילה בפני עצמם: "אי" בן שש תחנות שאינו נוגע בדבר *חייב* להוות קהילה נפרדת בכל חלוקה, וספירת כל אחד מהם מנפחת את מספר הקהילות מבלי לומר דבר על מבנה הרשת. מחברת 03 הראתה שהרכיב הגדול ביותר מכיל כמעט את כל התחנות, ולכן כמעט דבר אינו אובד.

זהו גם ההסבר הקונקרטי הראשון האפשרי לפער שבין 91 ל-73: ריצה על הגרף **המלא** מדווחת אוטומטית על `(מספר הרכיבים - 1)` קהילות יותר מריצה על הרכיב הגדול ביותר בלבד, עוד לפני שמעורבת בכך אקראיות כלשהי. התא מדפיס את גודלו המדויק של היסט זה עבור הגרף הנוכחי, כך שניתן לכמת את ההשפעה במקום לנחש אותה. תחנות שמחוץ לרכיב הגדול ביותר עדיין מיוצאות אל `community_assignments.csv`, עם מזהה קהילה `-1` שמשמעותו "לא נותחה".

In [ ]:
# --- Largest connected component ------------------------------------------
components = sorted(nx.connected_components(G), key=len, reverse=True)
Gc = G.subgraph(components[0]).copy()

outside = G.number_of_nodes() - Gc.number_of_nodes()
print(f'connected components in G          : {len(components):,}')
print(f'largest component (analysed)       : {Gc.number_of_nodes():,} nodes, '
      f'{Gc.number_of_edges():,} edges '
      f'({Gc.number_of_nodes() / G.number_of_nodes():.2%} of all stations)')
print(f'stations outside it (community -1) : {outside:,}')
print(f'running on the full graph instead would add {len(components) - 1} trivial communities')


## 7. Louvain: מה הוא ממקסם, והריצה הראשית

Louvain הוא היוריסטיקה חמדנית למקסום **modularity**

$$Q \;=\; \frac{1}{2m}\sum_{i,j}\left[A_{ij} - \gamma\,\frac{k_i k_j}{2m}\right]\delta(c_i, c_j)$$

כאשר $A_{ij}$ הוא משקל הנסיעות המסוכם בין התחנות $i$ ו-$j$, $k_i$ היא הדרגה הממושקלת של $i$, $m$ הוא משקל הקשתות הכולל, $\gamma$ היא הרזולוציה, ו-$\delta(c_i,c_j)$ שווה ל-1 כאשר שתי התחנות מצויות באותה קהילה. במילים: $Q$ משווה כמה משקל נופל בפועל **בתוך** הקהילות מול כמה משקל היה נופל בתוכן אילו אותן תחנות שמרו על דרגותיהן הממושקלות אך היו מחווטות באקראי. ערך $Q$ קרוב ל-0 משמעו "לא טוב יותר מאקראי"; ערכים שבטווח 0.3-0.7 בקירוב הם הסימן המקובל למבנה קהילתי אמיתי; המקסימום התיאורטי הוא 1.

האלגוריתם מחליף בין שני שלבים עד ש-$Q$ מפסיק להשתפר: (1) **הזזה מקומית** - סריקת הצמתים בסדר כלשהו והעברת כל צומת אל הקהילה השכנה המניבה את השיפור הגדול ביותר ב-modularity; (2) **צבירה (aggregation)** - כיווץ כל קהילה לצומת-על יחיד וחזרה על התהליך על הגרף הקטן יותר. שני השלבים זולים, ומשום כך Louvain מתמודד עם גרפים בסדר גודל זה בתוך שניות.

**החלק הסטוכסטי הוא שלב 1**: הסדר שבו נסרקים הצמתים ואופן ההכרעה בשוויונות מגיעים ממחולל המספרים האקראיים. סדרים שונים מובילים לאופטימומים מקומיים שונים של פונקציית מטרה שאינה קמורה, ואופטימומים אלה עשויים להיות בעלי modularity כמעט זהה ובכל זאת להיבדל במספר הקהילות שהם מכילים - בדרך כלל משום שקומץ אשכולות גבוליים מתמזגים או מתפצלים. זו בדיוק הסיבה שאנו מקבעים את `LOUVAIN_SEED`.

שלוש פונקציות עזר מבצעות את העבודה שלהלן. `louvain_partition` עוטפת את ה-backend הזמין מאחורי חתימה אחידה אחת המקבלת תמיד seed ורזולוציה מפורשים. `partition_modularity` מנקדת כל חלוקה באמצעות `networkx`, כך שערכי ה-modularity נשארים ברי-השוואה בין backends ולאורך הסריקות. `relabel_by_size` ממספרת מחדש את הקהילות כך שמזהה 0 הוא הגדולה ביותר, מזהה 1 השנייה בגודלה וכן הלאה, עם הכרעת שוויונות דטרמיניסטית - המזהים הגולמיים ש-Louvain מפיק הם שרירותיים והיו משתנים בין ריצות, דבר שהיה הופך את הטבלאות המיוצאות לבלתי ניתנות להשוואה.

In [ ]:
# --- Louvain helpers -------------------------------------------------------
def louvain_partition(graph, seed, resolution, weight=WEIGHT_ATTR):
    """One Louvain run -> {node: community_id}. Seed and resolution are always explicit."""
    if community_louvain is not None:
        return dict(community_louvain.best_partition(
            graph, weight=weight, resolution=resolution, random_state=seed))
    communities = nx.community.louvain_communities(
        graph, weight=weight, resolution=resolution, seed=seed)
    return {node: cid for cid, members in enumerate(communities) for node in members}


def groups_from_partition(partition):
    """{node: cid} -> {cid: set(nodes)}."""
    groups = defaultdict(set)
    for node, cid in partition.items():
        groups[cid].add(node)
    return groups


def partition_modularity(graph, partition, weight=WEIGHT_ATTR, resolution=1.0):
    """Newman-Girvan modularity of a partition, scored with networkx for comparability."""
    groups = groups_from_partition(partition)
    return float(nx.community.modularity(
        graph, list(groups.values()), weight=weight, resolution=resolution))


def relabel_by_size(partition):
    """Renumber communities so id 0 is the largest; ties broken deterministically."""
    groups = groups_from_partition(partition)
    order = sorted(groups.items(), key=lambda kv: (-len(kv[1]), min(str(n) for n in kv[1])))
    mapping = {old: new for new, (old, _) in enumerate(order)}
    return {node: mapping[cid] for node, cid in partition.items()}


# --- Headline run ----------------------------------------------------------
partition = relabel_by_size(louvain_partition(Gc, LOUVAIN_SEED, LOUVAIN_RESOLUTION))
modularity = partition_modularity(Gc, partition, resolution=LOUVAIN_RESOLUTION)

community_sizes = pd.Series(Counter(partition.values())).sort_values(ascending=False)
num_communities = int(community_sizes.size)

print(f'backend                : {LOUVAIN_BACKEND}')
print(f'seed / resolution      : {LOUVAIN_SEED} / {LOUVAIN_RESOLUTION}')
print(f'communities found      : {num_communities:,}')
print(f'modularity Q           : {modularity:.4f}')
print(f'largest community      : {int(community_sizes.iloc[0]):,} stations '
      f'({community_sizes.iloc[0] / Gc.number_of_nodes():.2%} of the component)')
print(f'median community size  : {community_sizes.median():.0f}')
print(f'communities with fewer than 10 stations: {int((community_sizes < 10).sum()):,}')


## 8. עד כמה יציב אותו מספר? סריקת seed, סריקת רזולוציה, והפער שבין 91 ל-73

המספר שהודפס לעיל הוא דגימה אחת מתוך התפלגות, לא קבוע. תא זה מריץ מחדש את Louvain על פני `SEED_SWEEP` (כשהרזולוציה מקובעת ב-`LOUVAIN_RESOLUTION`) ועל פני `RESOLUTION_SWEEP` (כשה-seed מקובע ב-`LOUVAIN_SEED`), ורושם עבור כל ריצה את מספר הקהילות, את ה-modularity, את גודל הקהילה הגדולה ביותר, ואת **מדד Rand המתוקנן (adjusted Rand index, ARI)** מול החלוקה הראשית.

ARI הוא העמודה המכרעת. הוא מודד באיזו תדירות שתי חלוקות מסכימות בשאלה אם *זוג* תחנות שייך יחד, בתיקון להסכמה הצפויה במקרה: 1.0 משמעו חלוקות זהות, 0.0 משמעו לא טוב יותר מאקראי. אם סריקת ה-seed מפיקה פיזור של מספרי קהילות אך ערכי ARI קרובים ל-1, המסקנה הכנה היא ש**החלוקה יציבה ורק המספר רועש** - קומץ אשכולות קטנים מתמזגים או מתפצלים בשוליים בעוד שהגושים האזוריים הגדולים נותרים במקומם. אילו ה-ARI היה נמוך, כל החלוקה הייתה בלתי יציבה ולא ניתן היה לסמוך על שום טענה ברמת הקהילה.

זו התשובה האמפירית לסתירה שבין הדוח למצגת. המנגנונים העשויים להזיז את מספר הקהילות, ואף אחד מהם אינו טעות, הם:

1. **ה-seed האקראי.** סדרי סריקת צמתים שונים מניבים אופטימומים מקומיים שונים של פונקציית מטרה לא-קמורה. מכומת ישירות בסריקת ה-seed שלהלן.
2. **הרזולוציה `gamma`.** המספר גדל עם `gamma`: העלאתה מפצלת קהילות, הורדתה ממזגת אותן. צינור עיבוד שהשאיר את הרזולוציה בברירת המחדל של ספרייה אחת בעוד שאחר השתמש בברירת מחדל שונה ידווח על מספר שונה. מכומת בסריקת הרזולוציה.
3. **איזה גרף חולק.** גרף מלא מול הרכיב הקשיר הגדול ביותר (סעיף 6 הדפיס את ההיסט המדויק); קשתות ממושקלות מול לא-ממושקלות; כיווץ הכיוונים ההפוכים על ידי חיבור מול לקיחת מקסימום. כל אלה משנים את פונקציית המטרה, ולפיכך את התשובה.
4. **איזה מימוש.** `python-louvain` והמימוש של `networkx` נבדלים בהכרעת שוויונות, בסף ההתכנסות שלהם, ובאופן שבו הם מעדנים את הגרף המצטבר; העידון של Leiden ב-igraph שונה עוד יותר. אותו שם אלגוריתם, אופטימום מקומי שונה. ה-backend שבו נעשה שימוש בפועל כאן מודפס בסעיף 2 ונשמר בקובץ ה-JSON המסכם.
5. **איזה snapshot של קובץ ה-GTFS.** בנייה מחדש של הגרף מקובץ עדכני יותר משנה מעט את קבוצות הצמתים והקשתות, ובכך מטלטלת את החלוקה.

הפרש של כמה עשרות קהילות עולה בקנה אחד לחלוטין עם פעולתם המשולבת של מנגנונים 1-4, ובפרט משום שרוב הקהילות הקרובות לגבול הפיצול/המיזוג הן קטנות. מה שראוי לצטט בדוח אינו המספר אלא ה-**modularity**, **התפלגות הגדלים** וה-**מבנה המרחבי** - שלושתם ניתנים לשחזור במידה רבה בהרבה, כפי שמראה הטבלה שלהלן.

In [ ]:
# --- Seed sweep and resolution sweep ---------------------------------------
from sklearn.metrics import adjusted_rand_score

node_order = list(Gc.nodes())
base_labels = [partition[n] for n in node_order]


def stability_row(experiment, seed, resolution):
    """Run Louvain once and describe the result relative to the headline partition."""
    p = louvain_partition(Gc, seed, resolution)
    sizes = Counter(p.values())
    return {
        'experiment': experiment,
        'seed': seed,
        'resolution': resolution,
        'num_communities': len(sizes),
        'modularity': round(partition_modularity(Gc, p, resolution=resolution), 4),
        'largest_community': max(sizes.values()),
        'communities_under_10': sum(1 for s in sizes.values() if s < 10),
        'ari_vs_headline': round(
            float(adjusted_rand_score(base_labels, [p[n] for n in node_order])), 4),
    }


stability = pd.DataFrame(
    [stability_row('seed sweep', s, LOUVAIN_RESOLUTION) for s in SEED_SWEEP]
    + [stability_row('resolution sweep', LOUVAIN_SEED, r) for r in RESOLUTION_SWEEP]
)
stability.to_csv(TABLES / 'louvain_stability.csv', index=False, encoding='utf-8-sig')

seed_runs = stability[stability['experiment'] == 'seed sweep']
spread = int(seed_runs['num_communities'].max() - seed_runs['num_communities'].min())
print('across seeds, at the fixed resolution:')
print(f"  communities : {int(seed_runs['num_communities'].min())} to "
      f"{int(seed_runs['num_communities'].max())}  (spread {spread})")
print(f"  modularity  : {seed_runs['modularity'].min():.4f} to {seed_runs['modularity'].max():.4f}")
print(f"  ARI vs headline : {seed_runs['ari_vs_headline'].min():.3f} to "
      f"{seed_runs['ari_vs_headline'].max():.3f}")
stability


### 8b. אותו ניסוי כאיור

הפאנל השמאלי משרטט את מספר הקהילות כפונקציה של ה-seed, כאשר ה-modularity של כל ריצה מודפס מעל העמודה שלה, כך שניתן לקרוא את השניים יחד: אם גובה העמודות משתנה בעוד שערכי ה-modularity כמעט אינם זזים, פונקציית המטרה שטוחה על פני הפתרונות הללו והמספר הוא כמעט שרירותי. הפאנל הימני משרטט את מספר הקהילות ואת ה-modularity כפונקציה של הרזולוציה `gamma` על צירים תאומים, ומראה שבחירה מכוונת של `gamma` מזיזה את המספר הרבה יותר מאשר האקראיות. קווי הייחוס המקווקווים מסמנים את הריצה הראשית.

In [ ]:
# --- figures/louvain_stability.png -----------------------------------------
seed_df = stability[stability['experiment'] == 'seed sweep'].sort_values('seed')
res_df = stability[stability['experiment'] == 'resolution sweep'].sort_values('resolution')

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

bars = axes[0].bar(seed_df['seed'].astype(str), seed_df['num_communities'], color='#2563eb')
for bar, q in zip(bars, seed_df['modularity']):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                 f'Q={q:.4f}', ha='center', va='bottom', fontsize=9)
axes[0].axhline(num_communities, color='#111827', linestyle='--', linewidth=1,
                label='headline run')
axes[0].set_xlabel('random seed')
axes[0].set_ylabel('number of communities')
axes[0].set_title(f'Community count across seeds (resolution = {LOUVAIN_RESOLUTION})')
axes[0].margins(y=0.15)
axes[0].legend()

axes[1].plot(res_df['resolution'], res_df['num_communities'], marker='o', color='#7c3aed')
axes[1].set_xlabel('resolution gamma')
axes[1].set_ylabel('number of communities', color='#7c3aed')
axes[1].tick_params(axis='y', labelcolor='#7c3aed')
axes[1].axvline(LOUVAIN_RESOLUTION, color='#111827', linestyle='--', linewidth=1)
twin = axes[1].twinx()
twin.plot(res_df['resolution'], res_df['modularity'], marker='s', color='#dc2626')
twin.set_ylabel('modularity Q', color='#dc2626')
twin.tick_params(axis='y', labelcolor='#dc2626')
twin.grid(False)
axes[1].set_title(f'Community count and modularity vs resolution (seed = {LOUVAIN_SEED})')

plt.tight_layout()
plt.savefig(FIGURES / 'louvain_stability.png', dpi=FIG_DPI)
plt.show()


## 9. חוות דעת שנייה: label propagation

הסקריפט המקורי הריץ גם **label propagation**, ואנו משמרים אותו כבדיקה צולבת. מדובר ברעיון שונה לחלוטין: כל צומת מתחיל בקהילה משלו, ולאחר מכן מאמץ שוב ושוב את התווית המוחזקת על ידי רוב שכניו, עד שאף צומת אינו מבקש להשתנות. הוא אינו ממקסם דבר במפורש - אין בו איבר modularity - ולכן הסכמה בין שתי השיטות היא ראיה בעלת משמעות לכך שהמבנה אמיתי ואינו תוצר לוואי של פונקציית המטרה של ה-modularity.

שתי הסתייגויות, המצוינות משום שהן חשובות לפירוש ההשוואה. ראשית, `label_propagation_communities` של `networkx` מתעלמת ממשקלי הקשתות, ולכן היא רואה טופולוגיה בלבד בעוד ש-Louvain רואה עוצמת שירות; שתי השיטות עונות על שאלות שונות במקצת. שנית, label propagation ידועה בכך שהיא מפיקה *גרעיניות* שונה מאוד - לעיתים קרובות קהילה אחת גדולה מאוד בתוספת זנב ארוך של קהילות קטנות - וזו תזכורת נוספת לכך ש"מספר הקהילות" הוא תכונה של השיטה לפחות באותה מידה שהוא תכונה של הרשת. אנו מנקדים את החלוקה שלה באמצעות אותה פונקציית modularity כדי ששתיהן יהיו על סקאלה בת-השוואה, ומשתמשים מחדש באותו מספור מחדש לפי סדר גודל.

In [ ]:
# --- Label propagation (unweighted, cross-check only) ----------------------
if RUN_LABEL_PROPAGATION:
    lp_communities = list(nx.community.label_propagation_communities(Gc))
    partition_lp = relabel_by_size(
        {node: cid for cid, members in enumerate(lp_communities) for node in members})
    modularity_lp = partition_modularity(Gc, partition_lp, resolution=LOUVAIN_RESOLUTION)
    lp_sizes = pd.Series(Counter(partition_lp.values())).sort_values(ascending=False)
    ari_lp = float(adjusted_rand_score(base_labels, [partition_lp[n] for n in node_order]))
    print(f'label propagation communities : {int(lp_sizes.size):,}')
    print(f'label propagation modularity  : {modularity_lp:.4f}   (Louvain: {modularity:.4f})')
    print(f'largest community             : {int(lp_sizes.iloc[0]):,} stations')
    print(f'agreement with Louvain (ARI)  : {ari_lp:.3f}')
else:
    partition_lp, modularity_lp, ari_lp = {}, None, None
    print('label propagation skipped (RUN_LABEL_PROPAGATION = False)')


## 10. ייצוא שיוך הקהילה ברמת התחנה

הראשונה מבין הטבלאות הנדרשות. שורה אחת לכל תחנה בגרף **כולו** - ולא רק ברכיב הגדול ביותר - הנושאת את זהותה, קואורדינטותיה, האזור, המטרופולין, הדרגה והקהילה שלה תחת כל אחד מן האלגוריתמים. תחנות שמחוץ לרכיב הגדול ביותר (וכן, אם label propagation דולג, כל תחנה בעמודת label propagation) מקבלות `-1`, ערך סמן שמשמעותו "לא שויכה בריצה זו" ואינו מזהה קהילה אמיתי לעולם. השארת התחנות שהוצאו מן הניתוח בתוך הקובץ, במקום השמטתן, מאפשרת למחברות הבאות לבצע join על רשימת התחנות המלאה בלי לאבד שורות בשקט.

In [ ]:
# --- tables/community_assignments.csv --------------------------------------
assign_df = pd.DataFrame([{
    'stop_id': n,
    'stop_name': G.nodes[n]['stop_name'],
    'lat': G.nodes[n]['lat'],
    'lon': G.nodes[n]['lon'],
    'region': G.nodes[n]['region'],
    'metro': G.nodes[n]['metro'],
    'degree': G.degree(n),
    'community_louvain': partition.get(n, -1),
    'community_lp': partition_lp.get(n, -1),
} for n in G.nodes()])
assign_df = assign_df.sort_values(['community_louvain', 'degree'],
                                  ascending=[True, False]).reset_index(drop=True)
assign_df.to_csv(TABLES / 'community_assignments.csv', index=False, encoding='utf-8-sig')

print(f"saved {TABLES / 'community_assignments.csv'}  ({len(assign_df):,} rows)")
print(f"unassigned stations (community -1): {int((assign_df['community_louvain'] == -1).sum()):,}")
assign_df.head(10)


## 11. תיאור כל קהילה - ותיוגה מתוך הנתונים

הטבלה הנדרשת השנייה. עבור כל קהילה אנו מחשבים, במעבר יחיד על הקשתות (כך שהעלות נותרת לינארית ולא ריבועית במספר הקהילות, כפי שהיה עולה מקריאת `subgraph` לכל קהילה בסקריפט המקורי):

* **גודל** - מספר התחנות, וחלקה מתוך הרכיב הנותח.
* **קשתות פנימיות / משקל פנימי** - מקטעים ונסיעות הנשארים בתוך הקהילה.
* **קשתות חיצוניות / משקל חיצוני** - מקטעים ונסיעות היוצאים ממנה. היחס `external_weight / (internal_weight + external_weight)` הוא מדד מסוג conductance לכמות ה"דליפה" של הקהילה: קרוב ל-0 משמעו אשכול עצמאי, קרוב ל-1 משמעו קבוצה המתקשרת בעיקר עם החוץ.
* **תחנות גבול** - כמה מתחנותיה בעלות שכן אחד לפחות בקהילה אחרת.
* **מרכז מסה (centroid)** - קו הרוחב וקו האורך הממוצעים על פני תחנותיה בעלות הקואורדינטות השמישות, והוא הקובע היכן תמוקם התווית על המפה.
* **אזור דומיננטי / מטרופולין דומיננטי** וחלקיהם - הערך השכיח של התכונות `region` ו-`metro` שמחברת 01 שייכה גיאוגרפית.
* **תחנת hub** - החברה בעלת הדרגה הממושקלת הגבוהה ביותר, כלומר התחנה העמוסה ביותר בקהילה.

ה-**תווית** נגזרת ואינה מקודדת קשיחות לעולם: היא המטרופולין השכיח של הקהילה עצמה מחובר לאזור השכיח שלה (עם הסרת כפילות כאשר השניים חופפים), והיא טובה בדיוק כמידת טיבה של הגיאוגרפיה ששויכה במעלה הזרם במחברת 01 - כלל גס של קו רוחב/קו אורך עם ארבע דיסקות מטרופוליניות, ולא מרשם מנהלי. העמודה `dominant_metro_share` מיוצאת דווקא כדי שהקורא יוכל לראות מתי תווית היא חלשה: קהילה שהמטרופולין השכיח שלה מכסה רק מחצית מתחנותיה היא תערובת אמיתית שתווית בת מילה אחת משטחת, ויש לקרוא אותה יחד עם שם תחנת ה-hub.

In [ ]:
# --- tables/community_summary.csv ------------------------------------------
groups = groups_from_partition(partition)

internal_edges = Counter(); internal_weight = Counter()
external_edges = Counter(); external_weight = Counter()
boundary_nodes = defaultdict(set)

for u, v, data in Gc.edges(data=True):
    cu, cv = partition[u], partition[v]
    w = data.get(WEIGHT_ATTR, 1)
    if cu == cv:
        internal_edges[cu] += 1
        internal_weight[cu] += w
    else:
        for c in (cu, cv):
            external_edges[c] += 1
            external_weight[c] += w
        boundary_nodes[cu].add(u)
        boundary_nodes[cv].add(v)

weighted_degree = dict(Gc.degree(weight=WEIGHT_ATTR))


def _mode(values):
    """Most common non-empty value and its share; ('', 0.0) when nothing is present."""
    counts = Counter(v for v in values if v)
    if not counts:
        return '', 0.0
    value, count = counts.most_common(1)[0]
    return value, round(count / len(values), 4)


def _label(metro, region):
    """Derived, data-driven community label: modal metro joined with modal region."""
    parts = [p for p in (metro, region) if p]
    return ' / '.join(dict.fromkeys(parts)) if parts else 'unlabelled'


rows = []
for cid, members in sorted(groups.items(), key=lambda kv: kv[0]):
    members = sorted(members)
    lats = [G.nodes[n]['lat'] for n in members if G.nodes[n]['lat'] is not None]
    lons = [G.nodes[n]['lon'] for n in members if G.nodes[n]['lon'] is not None]
    metro, metro_share = _mode([G.nodes[n]['metro'] for n in members])
    region, region_share = _mode([G.nodes[n]['region'] for n in members])
    hub = max(members, key=lambda n: (weighted_degree.get(n, 0), Gc.degree(n)))
    iw, ew = int(internal_weight[cid]), int(external_weight[cid])
    rows.append({
        'community_id': cid,
        'label': _label(metro, region),
        'size': len(members),
        'share_of_component': round(len(members) / Gc.number_of_nodes(), 5),
        'internal_edges': int(internal_edges[cid]),
        'external_edges': int(external_edges[cid]),
        'internal_weight': iw,
        'external_weight': ew,
        'external_weight_ratio': round(ew / (iw + ew), 4) if (iw + ew) else None,
        'boundary_stations': len(boundary_nodes[cid]),
        'boundary_share': round(len(boundary_nodes[cid]) / len(members), 4),
        'centroid_lat': round(float(np.mean(lats)), 4) if lats else None,
        'centroid_lon': round(float(np.mean(lons)), 4) if lons else None,
        'dominant_region': region,
        'dominant_region_share': region_share,
        'dominant_metro': metro,
        'dominant_metro_share': metro_share,
        'hub_stop_id': hub,
        'hub_stop_name': G.nodes[hub]['stop_name'],
    })

summary_df = pd.DataFrame(rows).sort_values('size', ascending=False).reset_index(drop=True)
summary_df.to_csv(TABLES / 'community_summary.csv', index=False, encoding='utf-8-sig')

print(f"saved {TABLES / 'community_summary.csv'}  ({len(summary_df):,} communities)")
print('communities per dominant metropolitan area (labels are derived, not hard-coded):')
print(summary_df['dominant_metro'].value_counts().to_string())
summary_df.head(TOP_N)[['community_id', 'label', 'size', 'internal_edges', 'external_edges',
                        'external_weight_ratio', 'dominant_metro_share', 'hub_stop_name']]


## 12. תחנות גשר בין-קהילתיות

הטבלה הנדרשת השלישית, והחלק המקשר מחברת זו בחזרה לשאלת החוסן (resilience) של הפרויקט. תחנה היא **גשר בין-קהילתי** כאשר לפחות אחד משכניה יושב בקהילה אחרת: זו נקודה שבה מסע נאלץ לצאת מאשכול אחד ולהיכנס לאחר. יש לשים לב שזהו מושג *שונה* מן הגשרים (bridges) במובן תורת הגרפים ממחברת 03 - אלה היו קשתות שהסרתן מנתקת את הגרף לחלוטין, בעוד שאלה תחנות הנושאות תנועה בין-אשכולית ואשר אובדנן היה כופה עקיפות ארוכות גם כאשר הרשת נותרת מקושרת מבחינה טכנית. תחנה יכולה להיות האחת, האחרת, שתיהן או אף אחת מהן, וכדאי לבצע join בין שתי הטבלאות.

עבור כל תחנה כזו אנו רושמים בכמה קהילות **שונות** נוגעת סביבתה (מדד המפתח של הסקריפט המקורי), כמה מן הקשתות הפוגעות בה חוצות גבול, את דרגתה ואת דרגתה הממושקלת, ומהו חלק התנועה שלה שהוא בין-קהילתי. דירוג לפי מספר הקהילות השונות תחילה ולפי משקל החצייה לאחר מכן מעלה אל פני השטח את התחנות שהן גם מרכזיות מבנית וגם עמוסות בשימוש - אלה שמחקר חוסן צריך להעמיד במבחן ראשונות. תחנה הנוגעת בקהילות רבות היא צומת מעבר בין מערכות אזוריות; תחנה הנוגעת בקהילה אחת נוספת בלבד אך נושאת משקל חצייה עצום היא מסדרון יחיד בעל נפח גבוה.

התא גם מצרף את שני הגדלים ברמת הרשת המעידים על מידת ההידוק שבין האשכולות: חלקן של הקשתות וחלקו של משקל הנסיעות הכולל החוצים גבול קהילה.

In [ ]:
# --- tables/inter_community_bridges.csv ------------------------------------
label_of = dict(zip(summary_df['community_id'], summary_df['label']))

bridge_rows = []
for node in Gc.nodes():
    own = partition[node]
    other_comms = set()
    crossing_edges = 0
    crossing_weight = 0
    for nb in Gc.neighbors(node):
        cid = partition[nb]
        if cid != own:
            other_comms.add(cid)
            crossing_edges += 1
            crossing_weight += Gc[node][nb].get(WEIGHT_ATTR, 1)
    if not other_comms:
        continue
    wdeg = weighted_degree.get(node, 0)
    bridge_rows.append({
        'stop_id': node,
        'stop_name': G.nodes[node]['stop_name'],
        'own_community': own,
        'own_community_label': label_of.get(own, ''),
        'connected_communities': len(other_comms),
        'crossing_edges': crossing_edges,
        'crossing_weight': int(crossing_weight),
        'degree': Gc.degree(node),
        'weighted_degree': int(wdeg),
        'crossing_weight_share': round(crossing_weight / wdeg, 4) if wdeg else None,
        'lat': G.nodes[node]['lat'],
        'lon': G.nodes[node]['lon'],
        'region': G.nodes[node]['region'],
        'metro': G.nodes[node]['metro'],
    })

bridges_df = (pd.DataFrame(bridge_rows)
              .sort_values(['connected_communities', 'crossing_weight'], ascending=False)
              .reset_index(drop=True))
bridges_df.to_csv(TABLES / 'inter_community_bridges.csv', index=False, encoding='utf-8-sig')

cross_edges = sum(1 for u, v in Gc.edges() if partition[u] != partition[v])
cross_weight = sum(d.get(WEIGHT_ATTR, 1) for u, v, d in Gc.edges(data=True)
                   if partition[u] != partition[v])
component_weight = sum(d.get(WEIGHT_ATTR, 1) for _, _, d in Gc.edges(data=True))

print(f"saved {TABLES / 'inter_community_bridges.csv'}  ({len(bridges_df):,} stations)")
print(f'boundary stations     : {len(bridges_df):,} of {Gc.number_of_nodes():,} '
      f'({len(bridges_df) / Gc.number_of_nodes():.2%} of the analysed component)')
print(f'cross-community edges : {cross_edges:,} of {Gc.number_of_edges():,} '
      f'({cross_edges / Gc.number_of_edges():.2%})')
print(f'cross-community trips : {cross_weight:,} of {component_weight:,} '
      f'({cross_weight / component_weight:.2%} of all trip weight)')
bridges_df.head(TOP_N)[['stop_id', 'stop_name', 'own_community_label', 'connected_communities',
                        'crossing_edges', 'crossing_weight', 'degree']]


## 13. מפת הקהילות

כל תחנה ברכיב הנותח משורטטת בקו האורך וקו הרוחב האמיתיים שלה, צבועה לפי קהילה; תחנות שמחוץ לרכיב משורטטות באפור בהיר כדי שהקורא יוכל לראות מה הוצא מן הניתוח. זהו האיור המכריע אם לחלוקה יש משמעות כלשהי: אם Louvain מצא מבנה אמיתי, הצבעים חייבים ליצור **כתמים רציפים מרחבית**, משום שקהילות של רשת תחבורה הן גיאוגרפיות מעצם הגדרתן - אוטובוסים מחברים מקומות הסמוכים זה לזה. פיזור אקראי דמוי קונפטי היה מעיד שהחלוקה היא רעש מספרי.

שני שינויים מכוונים ביחס לסקריפט המקורי. ראשית, הוא בנה את מפת הצבעים שלו באמצעות `tab20.resampled(n)`, שמבצעת *אינטרפולציה* בין עשרים הצבעים הקטגוריים ומייצרת עשרות גוונים עכורים וכמעט זהים ברגע שיש יותר קהילות מצבעים. אנו במקום זאת מחזוריים על פני שלוש מפות איכותניות בנות 20 צבעים (60 צבעים הנבדלים חזותית) מודולו מזהה הקהילה: הצבעים אכן חוזרים על עצמם, אך מכיוון שהמזהים מסודרים לפי גודל, הקהילות הגדולות השולטות בתמונה מקבלות כולן צבעים שונים. המפה נועדה להראות *רציפות*, ולא לאפשר לקורא לזהות קהילה מסוימת לפי צבעה - את התפקיד הזה ממלאות תוויות ההערה. שנית, המקור סינן קואורדינטות באמצעות `if lat and lon`, מה שמשמיט בשקט תחנה הנמצאת בדיוק ב-`0.0`, וגרוע מכך - משאיר ערכי `NaN`; אנו בודקים במפורש קיומו של ערך נוכח וסופי.

הקהילות הגדולות המעטות מסומנות בהערות במקומן, במרכזי המסה שלהן, עם התוויות הנגזרות מן הנתונים.

In [ ]:
# --- figures/community_map_louvain.png -------------------------------------
def has_coords(node):
    """True only when both coordinates are present and finite (0.0 included)."""
    lat, lon = G.nodes[node]['lat'], G.nodes[node]['lon']
    return (lat is not None and lon is not None
            and np.isfinite(lat) and np.isfinite(lon))


qualitative = [c for name in ('tab20', 'tab20b', 'tab20c')
               for c in matplotlib.colormaps[name].colors]

plot_nodes = [n for n in Gc.nodes() if has_coords(n)]
if not plot_nodes:
    raise ValueError('No station in the analysed component carries usable coordinates - '
                     'check nodes.csv from notebook 02.')
plot_lons = [G.nodes[n]['lon'] for n in plot_nodes]
plot_lats = [G.nodes[n]['lat'] for n in plot_nodes]
plot_cols = [qualitative[partition[n] % len(qualitative)] for n in plot_nodes]
bg_nodes = [n for n in G.nodes() if n not in partition and has_coords(n)]

fig, ax = plt.subplots(figsize=(8.5, 11))
if bg_nodes:
    ax.scatter([G.nodes[n]['lon'] for n in bg_nodes], [G.nodes[n]['lat'] for n in bg_nodes],
               s=MAP_POINT_SIZE, color='#cbd5e1', alpha=0.6, linewidths=0,
               label=f'outside largest component (n={len(bg_nodes):,})')
ax.scatter(plot_lons, plot_lats, s=MAP_POINT_SIZE, color=plot_cols,
           alpha=MAP_ALPHA, linewidths=0)

for _, row in summary_df.head(ANNOTATE_TOP).iterrows():
    if row['centroid_lat'] is None or row['centroid_lon'] is None:
        continue
    ax.annotate(f"{row['label']}\n(n={int(row['size']):,})",
                (row['centroid_lon'], row['centroid_lat']),
                fontsize=8, fontweight='bold', ha='center', zorder=6,
                bbox=dict(boxstyle='round,pad=0.25', facecolor='white', alpha=0.75,
                          edgecolor='#94a3b8', linewidth=0.5))

ax.set_aspect(1 / np.cos(np.deg2rad(float(np.mean(plot_lats)))))
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Louvain communities of the Israeli public-transport network\n'
             f'{num_communities:,} communities, modularity Q = {modularity:.4f} '
             f'(seed {LOUVAIN_SEED}, resolution {LOUVAIN_RESOLUTION})')
if bg_nodes:
    ax.legend(loc='upper left', fontsize=8, markerscale=3)
plt.tight_layout()
plt.savefig(FIGURES / 'community_map_louvain.png', dpi=FIG_DPI)
plt.show()

print(f'plotted {len(plot_nodes):,} of {Gc.number_of_nodes():,} analysed stations '
      f'({Gc.number_of_nodes() - len(plot_nodes):,} lack usable coordinates)')


## 14. התפלגות גדלי הקהילות

שני מבטים על אופן חלוקת התחנות. ה**פאנל השמאלי** מדרג את הקהילות הגדולות ביותר לפי גודל, מתויגות בתווית הנגזרת מן הנתונים ובתחנה העמוסה ביותר של כל אחת, כך שהקורא יראה מיד אם החלוקה נשלטת על ידי מספר גושים מטרופוליניים. ה**פאנל הימני** מציג את התפלגות דרגה-גודל כולה על צירים לוגריתמיים - כל הקהילות, לא רק המובילות - וזהו המבט הכן: הוא חושף אם קיים זנב ארוך של קהילות זעירות ובאיזה קצב בערך יורד הגודל עם הדרגה.

הזנב חשוב לפירוש סעיף 8. קהילות בנות קומץ תחנות הן בדיוק אלה המתמזגות או מתפצלות כאשר ה-seed משתנה, ולכן זנב שמן של קהילות זעירות הוא הסבר חזותי ישיר לכך שהמספר הכולל אינו יציב בעוד שהגושים הגדולים כן. קו הייחוס האופקי מסמן גודל של עשר תחנות, הסף המדווח בהדפסת המפתח של סעיף 7.

In [ ]:
# --- figures/community_size_distribution.png -------------------------------
top = summary_df.head(TOP_N).iloc[::-1]
top_labels = [f'#{int(cid)} {lab} - {hub}' for cid, lab, hub
              in zip(top['community_id'], top['label'], top['hub_stop_name'])]

sizes_sorted = summary_df['size'].to_numpy()
ranks = np.arange(1, len(sizes_sorted) + 1)

fig, axes = plt.subplots(1, 2, figsize=(15, 8))

bars = axes[0].barh(range(len(top)), top['size'].to_numpy(), color='#7c3aed')
axes[0].set_yticks(range(len(top)))
axes[0].set_yticklabels(top_labels, fontsize=8)
for bar, val in zip(bars, top['size'].to_numpy()):
    axes[0].text(bar.get_width(), bar.get_y() + bar.get_height() / 2,
                 f' {int(val):,}', va='center', fontsize=8)
axes[0].set_xlabel('Number of stations')
axes[0].set_title(f'Largest {len(top)} communities of {num_communities:,}')
axes[0].margins(x=0.14)

axes[1].scatter(ranks, sizes_sorted, s=18, color='#2563eb', alpha=0.75)
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].axhline(10, color='#dc2626', linestyle='--', linewidth=1, label='size = 10 stations')
axes[1].set_xlabel('Community rank (log scale)')
axes[1].set_ylabel('Number of stations (log scale)')
axes[1].set_title('Rank-size distribution of all communities')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES / 'community_size_distribution.png', dpi=FIG_DPI)
plt.show()

print(summary_df['size'].describe().to_string())


## 15. היכן נמצאות תחנות הגבול

האיור האחרון ממקם על המפה את תחנות הגשר הבין-קהילתיות המדורגות גבוה ביותר, כשגודלן וצבען נקבעים לפי מספר הקהילות שהן נוגעות בהן, על רקע חיוור של הרשת כולה. מכיוון שקהילות ברשת תחבורה הן גיאוגרפיות, נקודות אלה אמורות ליפול על *התפרים* שבין הכתמים הצבעוניים של סעיף 13 - המסדרונות המקשרים בין אשכול מטרופוליני אחד למשנהו. תפרים אלה הם המקומות שבהם כשל אינו רק מעכב מסע אלא כופה עליו לצאת מן האשכול שלו כליל, ומשום כך טבלה זו מזינה את מחברות החוסן.

In [ ]:
# --- figures/inter_community_bridges_map.png -------------------------------
top_bridges = bridges_df.dropna(subset=['lat', 'lon']).head(100)

fig, ax = plt.subplots(figsize=(8.5, 11))
bg = [n for n in G.nodes() if has_coords(n)]
ax.scatter([G.nodes[n]['lon'] for n in bg], [G.nodes[n]['lat'] for n in bg],
           s=1, color='#e2e8f0', alpha=0.5, linewidths=0)
sc = ax.scatter(top_bridges['lon'], top_bridges['lat'],
                s=top_bridges['connected_communities'] * 20,
                c=top_bridges['connected_communities'], cmap='Reds',
                edgecolors='black', linewidths=0.3, zorder=5)
plt.colorbar(sc, ax=ax, label='distinct communities touched')
ax.set_aspect(1 / np.cos(np.deg2rad(float(np.mean(plot_lats)))))
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'Top {len(top_bridges)} inter-community bridge stations\n'
             '(marker size = number of communities the station connects)')
plt.tight_layout()
plt.savefig(FIGURES / 'inter_community_bridges_map.png', dpi=FIG_DPI)
plt.show()


## 16. סטטיסטיקות המפתח

כל מה שראוי לצטט משלב זה, מרוכז במילון אחד ונכתב הן כ-JSON והן כקובץ CSV בן שורה אחת. כל שדה מחושב לעיל - וה-seed, הרזולוציה וה-backend נשמרים *לצד* התוצאות דווקא כדי שקורא עתידי יוכל לדעת איזו תצורה הפיקה אותן, וזו המשמעת שהיעדרה גרם מלכתחילה לבלבול שבין 91 ל-73.

In [ ]:
# --- community_detection_summary.json --------------------------------------
summary = {
    'louvain_backend': LOUVAIN_BACKEND,
    'louvain_seed': LOUVAIN_SEED,
    'louvain_resolution': LOUVAIN_RESOLUTION,
    'weighted': True,
    'graph_nodes': G.number_of_nodes(),
    'graph_edges': G.number_of_edges(),
    'connected_components': len(components),
    'analysed_nodes': Gc.number_of_nodes(),
    'analysed_edges': Gc.number_of_edges(),
    'num_communities': num_communities,
    'modularity': round(modularity, 4),
    'largest_community_size': int(community_sizes.iloc[0]),
    'largest_community_share': round(float(community_sizes.iloc[0]) / Gc.number_of_nodes(), 4),
    'median_community_size': float(community_sizes.median()),
    'communities_under_10_stations': int((community_sizes < 10).sum()),
    'cross_community_edges': int(cross_edges),
    'cross_community_edge_share': round(cross_edges / Gc.number_of_edges(), 4),
    'cross_community_weight_share': round(cross_weight / component_weight, 4),
    'inter_community_stations': int(len(bridges_df)),
    'inter_community_station_share': round(len(bridges_df) / Gc.number_of_nodes(), 4),
    'max_communities_touched_by_one_station':
        int(bridges_df['connected_communities'].max()) if len(bridges_df) else 0,
    'seed_sweep_seeds': list(SEED_SWEEP),
    'seed_sweep_min_communities': int(seed_runs['num_communities'].min()),
    'seed_sweep_max_communities': int(seed_runs['num_communities'].max()),
    'seed_sweep_min_modularity': float(seed_runs['modularity'].min()),
    'seed_sweep_max_modularity': float(seed_runs['modularity'].max()),
    'seed_sweep_min_ari_vs_headline': float(seed_runs['ari_vs_headline'].min()),
    'label_propagation_communities': int(len(set(partition_lp.values()))) if partition_lp else None,
    'label_propagation_modularity': round(modularity_lp, 4) if modularity_lp is not None else None,
    'label_propagation_ari_vs_louvain': round(ari_lp, 4) if ari_lp is not None else None,
}

with open(STAGE / 'community_detection_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
flat = {k: (json.dumps(v) if isinstance(v, list) else v) for k, v in summary.items()}
pd.DataFrame([flat]).to_csv(TABLES / 'community_detection_summary.csv',
                            index=False, encoding='utf-8-sig')

print('saved:', STAGE / 'community_detection_summary.json')
for p in sorted(TABLES.iterdir()) + sorted(FIGURES.iterdir()):
    print(f'  {p.relative_to(STAGE)}  ({p.stat().st_size / 1024:,.0f} KB)')
pd.DataFrame({'metric': list(summary.keys()), 'value': [str(v) for v in summary.values()]})


## מסקנות

* **הרשת אכן מתפרקת לקהילות, וערך ה-modularity הוא הטענה שראוי לטעון.** הריצה הראשית מדפיסה את ה-modularity שלה, `Q`, בסעיף 7; יש לקרוא אותו מול אמת המידה המקובלת שלפיה ערכים שבטווח 0.3-0.7 בקירוב מעידים על מבנה קהילתי אמיתי, בעוד שערך קרוב ל-0 היה משמעו שהחלוקה אינה טובה יותר ממקרה. ה-modularity, ולא מספר הקהילות, הוא הגודל הניתן לשחזור - הוא כמעט לא זז לאורך סריקת ה-seed.

* **מספר הקהילות אינו תכונה של הרשת.** זו התשובה הישירה למחלוקת שבין 91 ל-73, בין הדוח המסכם בעברית לבין צינור העיבוד של המצגת. Louvain הוא היוריסטיקה חמדנית סטוכסטית מעל פונקציית מטרה לא-קמורה, ולכן ה-seed לבדו מזיז את המספר בעוד שה-modularity נותר שטוח במהותו ומדד Rand המתוקנן מול הריצה הראשית נותר גבוה: החלוקה יציבה, רק הרישום בשוליה אינו. נוסף על כך, הרזולוציה `gamma` מזיזה את המספר במכוון ובמידה רבה בהרבה; הרצה על הגרף המלא במקום על הרכיב הקשיר הגדול ביותר מוסיפה קהילה אחת לכל רכיב מבודד (ההיסט המדויק מודפס בסעיף 6); ו-`python-louvain`, `networkx` ו-igraph/Leiden מגיעים לאופטימומים מקומיים שונים מקלט זהה. כל אחד משני המספרים הקודמים ניתן לשחזור תחת צירוף כלשהו של הגדרות אלה, ולכן מחברת זו מקבעת את ה-seed ואת הרזולוציה, רושמת את ה-backend, ומדווחת על תוצאתה שלה במקום לצטט מספר הזכור מן העבר. **הדרך הנכונה לצטט תוצאת Louvain היא "N קהילות ב-seed S, רזולוציה R, מימוש X, modularity Q", ולעולם לא "N קהילות".**

* **הקהילות גיאוגרפיות, וזו בדיקת השפיות המשמעותית.** צבעי המפה יוצרים כתמים מרחביים רציפים ולא קונפטי, והתוויות הנגזרות - המטרופולין השכיח והאזור השכיח של כל קהילה, המחושבים בזמן ריצה - מתיישבות עם המבנה המטרופוליני של המדינה. זהו הצפוי מרשת תחבורה, שבה השכנות מוגבלת על ידי מרחק פיזי, וזו הראיה המרכזית לכך שהחלוקה בעלת משמעות ואינה רעש מספרי. ההסתייגות היא שתוויות אלה יורשות את כלל האזורים הגס של מחברת 01, המבוסס על קו רוחב/קו אורך, ואת ארבע הדיסקות המטרופוליניות שלה, ולכן ערך `dominant_metro_share` נמוך ב-`community_summary.csv` מסמן קהילה שתוויתה בת המילה האחת היא באמת פישוט.

* **התפלגות הגדלים בלתי אחידה מאוד, והזנב מסביר את חוסר היציבות.** כמה קהילות מטרופוליניות גדולות מחזיקות את רוב התחנות, בעוד שזנב ארוך של קהילות קטנות מכסה את הפריפריה, כפי שמראה פאנל דרגה-גודל. אותן קהילות קטנות הן בדיוק אלה המתמזגות או מתפצלות בין seeds, ולכן הזנב השמן והמספר הבלתי יציב הם אותה תופעה עצמה הנראית פעמיים. מכאן גם שסטטיסטיקות מסכמות על פני קהילות - למשל גודל קהילה ממוצע - הן כמעט חסרות משמעות כאן; יש להציג את ההתפלגות, לא למצע אותה.

* **תחנות הגבול הן מיעוט, וזו ממצא החוסן.** רק חלק מן התחנות הן בעלות שכן כלשהו בקהילה אחרת, ורק חלק ממשקל הנסיעות חוצה גבול קהילה; החלקים המדויקים מודפסים בסעיף 12 ונשמרים בקובץ ה-JSON המסכם. רשת שאשכוליה מחוברים באמצעות מספר קטן יחסית של תחנות היא יעילה אך חשופה: תחנות אלה נושאות את התנועה הבין-אזורית, ואלה הנוגעות במרב הקהילות הן המועמדות הטבעיות הראשונות לניסויי התקיפה הממוקדת במחברות העמידות (robustness). יש לשים לב שאלה *אינם* אותם אובייקטים כמו הגשרים (bridges) של מחברת 03 - אלה היו קשתות שהסרתן מנתקת את הגרף, ואלה תחנות הנושאות זרימה בין-אשכולית - וביצוע join בין שתי הטבלאות מזהה את התחנות שהן גם וגם.

* **מגבלות, בכנות.** החלוקה נגזרת מ*ספירת נסיעות מתוכננות ב-snapshot יחיד של GTFS*, ללא עומסי נוסעים, ללא לוח זמנים, ללא זמני מעבר וללא קישורי הליכה בין תחנות סמוכות - שתי תחנות משני צידיו של אותו צומת הן צמתים נפרדים אלא אם כן נסיעה מחברת ביניהן, מה שמקטע את הגרף באופן שנוסעים אמיתיים אינם חווים. Louvain סובל גם ממגבלת הרזולוציה הידועה: ב-`gamma = 1` הוא אינו מסוגל לזהות קהילות קטנות בהרבה מן השורש הריבועי של משקל הקשתות הכולל, ולכן אשכולות מקומיים קטנים באמת עלולים להיבלע בשכניהם הגדולים. Label propagation, שאינה ממקסמת דבר ומתעלמת ממשקלים, מדווחת לצידו כבדיקה בלתי תלויה ולא כמתחרה: היכן ששתי השיטות מסכימות, ניתן לדבר על המבנה בביטחון; היכן שהן חלוקות, אין להפריז בפירוש הגרעיניות של שיטה בודדת כלשהי.